In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import cv2
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Dense,
    Dropout,
    Flatten,
    BatchNormalization
    )

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from tensorflow.keras.preprocessing.image import ImageDataGenerator

from tensorflow.keras.utils import to_categorical 

print("Tensorflow version", tf.__version__)

Tensorflow version 2.10.1


In [2]:
import os

In [3]:
# Define paths

train_path = r'E:\MY_DATASET\train'
test_path = r'E:\MY_DATASET\test'

In [4]:
# Emotion Labels

emotion_labels = {
    0: 'angry',
    1: 'disgust',
    2: 'fear',
    3: 'happy',
    4: 'neutral',
    5: 'sad',
    6: 'surprise'
}

In [5]:
# Loading Datasets


def load_dataset(dataset_path):
    images = []
    labels = []
    
    for emotion_num, emotion_name in emotion_labels.items():
        emotion_path = os.path.join(dataset_path, emotion_name)
        
        # Check if directory exists
        
        if not os.path.isdir(emotion_path):
            print(f"Warning: Directory {emotion_path} not found. Skipping...")
            continue
        
        for img_file in os.listdir(emotion_path):
            if img_file.endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(emotion_path, img_file)
                
                try:
                    # Read and preprocess image
                    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                    img = cv2.resize(img, (48,48))
                    
                    # Normalize pixel values
                    img = img / 255.0
                    
                    images.append(img)
                    labels.append(emotion_num)
                    
                except Exception as e:
                    print(f"Error loading image {img_path} : {e}")
                    
                    print(f"Error loading image {img_path}: {e}")

    # Convert to numpy arrays
    images = np.array(images)
    labels = np.array(labels)

    return images, labels

# Load training and test data
print("Loading training data...")
train_images, train_labels = load_dataset(train_path)
print("Loading test data...")
test_images, test_labels = load_dataset(test_path)

# Reshape images for CNN input
train_images = train_images.reshape(-1, 48, 48, 1)
test_images = test_images.reshape(-1, 48, 48, 1)

# One-hot encode labels
train_labels = to_categorical(train_labels, num_classes=7)
test_labels = to_categorical(test_labels, num_classes=7)

# Split into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    train_images,
    train_labels,
    test_size=0.2,
    random_state=42,
    stratify=train_labels  # Maintain class distribution
)

print("\nDataset Statistics:")
print(f"Training set shape: {X_train.shape} ({(X_train.shape[0]/len(train_images)*100):.1f}% of total)")
print(f"Validation set shape: {X_val.shape} ({(X_val.shape[0]/len(train_images)*100):.1f}% of total)")
print(f"Test set shape: {test_images.shape}")

Loading training data...
Loading test data...

Dataset Statistics:
Training set shape: (22967, 48, 48, 1) (80.0% of total)
Validation set shape: (5742, 48, 48, 1) (20.0% of total)
Test set shape: (7178, 48, 48, 1)


In [6]:
# Create data generator for augmentation

train_datagen = ImageDataGenerator(
    rotation_range = 15,
    width_shift_range = 0.1,
    height_shift_range = 0.1,
    shear_range = 0.1,
    zoom_range = 0.1,
    horizontal_flip = True,
    fill_mode = 'nearest'
)

train_datagen.fit(X_train)

In [7]:
def create_model():
    model = Sequential()
    
    # First Conv Block
    model.add(Conv2D(64, (3,3), padding = 'same', activation= 'relu', input_shape = (48,48,1)))
    model.add(BatchNormalization())
    model.add(Conv2D(64, (3,3), padding = 'same', activation= 'relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size = (2,2)))
    model.add(Dropout(0.25))
    
    # Second Conv Block
    model.add(Conv2D(128, (3,3), padding = 'same', activation= 'relu', ))
    model.add(BatchNormalization())
    model.add(Conv2D(128, (3,3), padding = 'same', activation= 'relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size = (2,2)))
    model.add(Dropout(0.25))
    
    # Third Conv Block
    model.add(Conv2D(256, (3,3), padding = 'same', activation= 'relu', ))
    model.add(BatchNormalization())
    model.add(Conv2D(256, (3,3), padding = 'same', activation= 'relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size = (2,2)))
    model.add(Dropout(0.25))
    
    # Fourth Conv Block
    model.add(Flatten())
    model.add(Dense(256, activation = 'relu' ))
    model.add(BatchNormalization())
    model.add(Dropout(0.25))
    model.add(Dense(512, activation= 'relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.25))
    model.add(Dense(7, activation= 'softmax'))
    
    return model

model = create_model()
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 48, 48, 64)        640       
                                                                 
 batch_normalization (BatchN  (None, 48, 48, 64)       256       
 ormalization)                                                   
                                                                 
 conv2d_1 (Conv2D)           (None, 48, 48, 64)        36928     
                                                                 
 batch_normalization_1 (Batc  (None, 48, 48, 64)       256       
 hNormalization)                                                 
                                                                 
 max_pooling2d (MaxPooling2D  (None, 24, 24, 64)       0         
 )                                                               
                                                        

In [8]:
# Define Optimizer

optimizer = Adam(learning_rate=0.0001)


In [9]:
# Compile the model 

model.compile(optimizer= optimizer,
              loss = 'categorical_crossentropy',
              metrics = ['accuracy'])

In [10]:
# Callbacks

checkpoint = ModelCheckpoint(
    'best_model.h5',
    monitor= 'val_accuracy',
    save_best_only= True,
    mode= 'max',
    verbose=1
)

early_stopping = EarlyStopping(
    monitor = 'val_loss',
    patience= 15,
    restore_best_weights= True,
    verbose= 1
)

reduce_lr = ReduceLROnPlateau(
    monitor = 'val_loss',
    patience= 5,
    factor = 0.2,
    min_lr = 0.00001,
    verbose = 1    
)

In [11]:
from tensorflow.keras.models import load_model

model = load_model('best_model.h5')

# Training parameters

batch_size = 64
epochs = 100

# Train the models

# history = model.fit(
#     train_datagen.flow(X_train, y_train, batch_size = batch_size),
#     steps_per_epoch = len(X_train) // batch_size,
#     epochs = epochs,
#     validation_data = (X_val, y_val),
#     callbacks = [checkpoint, early_stopping, reduce_lr]
# )

history = model.fit(
    train_datagen.flow(X_train, y_train, batch_size=batch_size),
    steps_per_epoch=len(X_train) // batch_size,
    epochs=100,
    initial_epoch=97,   # important
    validation_data=(X_val, y_val),
    callbacks=[checkpoint, early_stopping, reduce_lr]
)


Epoch 98/100
358/358 [==============================] - ETA: 0s - loss: 0.8711 - accuracy: 0.6751
Epoch 98: val_accuracy improved from -inf to 0.64943, saving model to best_model.h5
358/358 [==============================] - 392s 1s/step - loss: 0.8711 - accuracy: 0.6751 - val_loss: 0.9774 - val_accuracy: 0.6494 - lr: 1.0000e-05
Epoch 99/100
358/358 [==============================] - ETA: 0s - loss: 0.8707 - accuracy: 0.6743
Epoch 99: val_accuracy improved from 0.64943 to 0.64995, saving model to best_model.h5
358/358 [==============================] - 391s 1s/step - loss: 0.8707 - accuracy: 0.6743 - val_loss: 0.9761 - val_accuracy: 0.6499 - lr: 1.0000e-05
Epoch 100/100
358/358 [==============================] - ETA: 0s - loss: 0.8738 - accuracy: 0.6739
Epoch 100: val_accuracy improved from 0.64995 to 0.65169, saving model to best_model.h5
358/358 [==============================] - 392s 1s/step - loss: 0.8738 - accuracy: 0.6739 - val_loss: 0.9715 - val_accuracy: 0.6517 - lr: 1.0000e-05

In [13]:
import matplotlib.pyplot as plt

# Load the best saved model
model.load_weights('best_model.h5')

# Evaluate on test set
test_loss, test_acc = model.evaluate(test_images, test_labels)
print(f'Test Accuracy: {test_acc*100:.2f}%')

# Generate predictions
y_pred = model.predict(test_images)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(test_labels, axis=1)

# Classification report
print(classification_report(y_true, y_pred_classes, target_names=['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']))

# Confusion matrix
plt.figure(figsize=(10,8))
sns.heatmap(confusion_matrix(y_true, y_pred_classes), 
            annot=True, 
            fmt='d', 
            cmap='Blues',
            xticklabels=['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral'],
            yticklabels=['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral'])
plt.title('Confusion Matrix')
plt.show()

225/225 [==============================] - 29s 129ms/step - loss: 0.9556 - accuracy: 0.6542
Test Accuracy: 65.42%
225/225 [==============================] - 28s 125ms/step
              precision    recall  f1-score   support

       Angry       0.58      0.58      0.58       958
     Disgust       0.65      0.46      0.54       111
        Fear       0.59      0.37      0.45      1024
       Happy       0.86      0.87      0.86      1774
         Sad       0.53      0.73      0.61      1233
    Surprise       0.56      0.48      0.52      1247
     Neutral       0.74      0.81      0.77       831

    accuracy                           0.65      7178
   macro avg       0.64      0.61      0.62      7178
weighted avg       0.66      0.65      0.65      7178



AttributeError: 'RcParams' object has no attribute '_get'

In [ ]:
# Plot accuracy and loss
plt.figure(figsize=(14,5))

# Accuracy
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Loss
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.show()